# My investigations: what does each part of NSGA-II do?
**DCS 340 · Max Hoffman**

> [Your question in your own words: e.g. "Which part of NSGA-II matters most for getting a good front?"]

Two measures for each run, on the true front of the test problem:
- **Coverage**: fraction of the f1 axis the final population spans (1.0 = whole front).
- **Gap**: average vertical distance from each point to the true front (0 = on the front).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# --- NSGA-II implementation (same as src/nsga2.py) ---
"""NSGA-II from scratch (numpy only), as used in the DCS 340 class activity.

Reference: Deb, Pratap, Agarwal & Meyarivan (2002), IEEE TEC 6(2):182-197.
Test problems: Zitzler, Deb & Thiele (2000), Evolutionary Computation 8(2).
"""
import numpy as np
rng = np.random.default_rng(0)

def zdt1(X):
    f1 = X[:, 0]
    g = 1 + 9 * X[:, 1:].mean(axis=1)
    f2 = g * (1 - np.sqrt(f1 / g))
    return np.c_[f1, f2]

def nondominated_sort(F):
    n = len(F)
    dominated_by = [[] for _ in range(n)]   # who i dominates
    counts = np.zeros(n, int)                # how many dominate i
    for i in range(n):
        le = np.all(F[i] <= F, axis=1) & np.any(F[i] < F, axis=1)   # i dominates j
        dominated_by[i] = list(np.where(le)[0])
        counts[i] = np.sum(np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1))
    fronts, current = [], list(np.where(counts == 0)[0])
    while current:
        fronts.append(current)
        nxt = []
        for i in current:
            for j in dominated_by[i]:
                counts[j] -= 1
                if counts[j] == 0:
                    nxt.append(j)
        current = nxt
    return fronts

def crowding_distance(F):
    n, m = F.shape
    d = np.zeros(n)
    for k in range(m):
        order = np.argsort(F[:, k])
        d[order[0]] = d[order[-1]] = np.inf          # always keep the extremes
        span = F[order[-1], k] - F[order[0], k] or 1.0
        d[order[1:-1]] += (F[order[2:], k] - F[order[:-2], k]) / span
    return d

def vary(P, eta=15, pm=None):
    # simple variation: uniform crossover + gaussian mutation
    n, d = P.shape
    pm = pm or 1 / d
    idx = rng.integers(0, n, (n, 2))
    a, b = P[idx[:, 0]], P[idx[:, 1]]
    u = rng.random((n, d))
    C = np.where(u < 0.5, a, b) + rng.normal(0, 0.1, (n, d)) * (rng.random((n, d)) < pm)
    return np.clip(C, 0, 1)

def nsga2(problem, n_var=30, pop=100, gens=150, use_crowding=True, seed=0):
    global rng
    rng = np.random.default_rng(seed)
    P = rng.random((pop, n_var))
    for _ in range(gens):
        R = np.vstack([P, vary(P)])              # parents + children (elitism)
        F = problem(R)
        survivors = []
        for front in nondominated_sort(F):
            if len(survivors) + len(front) <= pop:
                survivors += front
            else:
                need = pop - len(survivors)
                if use_crowding:
                    cd = crowding_distance(F[front])
                    pick = np.argsort(-cd)[:need]
                else:
                    pick = rng.choice(len(front), need, replace=False)   # no diversity pressure
                survivors += list(np.array(front)[pick])
                break
        P = R[survivors]
    return problem(P)


def zdt2(X):
    f1 = X[:, 0]
    g = 1 + 9 * X[:, 1:].mean(axis=1)
    f2 = g * (1 - (f1 / g) ** 2)
    return np.c_[f1, f2]



In [ ]:
def measure(F, problem):
    true_f2 = 1 - np.sqrt(F[:, 0]) if problem is zdt1 else 1 - F[:, 0] ** 2
    return np.ptp(F[:, 0]), float(np.mean(F[:, 1] - true_f2))

runs = {
    'baseline':        dict(),
    'no crowding':     dict(use_crowding=False),
    'pop = 10':        dict(pop=10),
    'gens = 20':       dict(gens=20),
    'seed = 3':        dict(seed=3),
    'ZDT2 (concave)':  dict(problem=zdt2),
}
results = {}
for name, kw in runs.items():
    kw = dict(kw); p = kw.pop('problem', zdt1)
    F = nsga2(p, **kw)
    results[name] = (F, p, *measure(F, p))
    print(f'{name:16s} coverage {results[name][2]:.2f}   gap {results[name][3]:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7.5), sharex=True, sharey=True)
t = np.linspace(0, 1, 200)
for ax, (name, (F, p, cov, gap)) in zip(axes.flat, results.items()):
    ax.plot(t, 1 - np.sqrt(t) if p is zdt1 else 1 - t**2, color='#cfdad2', lw=4)
    ax.scatter(F[:, 0], F[:, 1], s=12, color='#2E7D4F')
    ax.set_title(f'{name}\ncoverage {cov:.2f}, gap {gap:.3f}', fontsize=10)
    ax.set_ylim(-0.05, 2)
import os
out = '../figures' if os.path.isdir('../figures') else '.'
plt.tight_layout(); plt.savefig(f'{out}/investigations.png', dpi=150); plt.show()

## What I found
[Your own words. Prompts:
- Which change hurt the most? Was it the one you predicted?
- Crowding off: the front is reached but not spread. What does that tell you about which step does what?
- pop=10 vs gens=20: both fall short of the front. Same reason or different?
- ZDT2 has a sagging front. Compare with the weighted-sum failure in the class notebook.]

## Seed check (is the baseline just lucky?)

In [ ]:
covs = [measure(nsga2(zdt1, seed=s), zdt1)[0] for s in range(5)]
nc   = [measure(nsga2(zdt1, seed=s, use_crowding=False), zdt1)[0] for s in range(5)]
print('coverage with crowding, 5 seeds:   ', np.round(covs, 2))
print('coverage without crowding, 5 seeds:', np.round(nc, 2))

[Your interpretation of the seed check.]

## References
- Deb et al. (2002), NSGA-II, IEEE TEC 6(2).
- Zitzler, Deb & Thiele (2000), ZDT test problems, Evolutionary Computation 8(2).